In [1]:
#!/usr/bin/env python3
"""
End-to-end training script for blind array-calibration
(Ψ gains and Φ phases) from sample-covariance matrices.

• Loads the NPZ dataset whose path you give in DATA_PATH.
• Infers the array size M automatically from the file.
• Builds a lightweight CNN → dense head that outputs 2M values.
• Trains with an 80 / 10 / 10 split, shows RMSE per epoch.
• Saves the trained weights as calibnet_M<whatever>.pt
"""

# -------------------------------------------------
# 0)  Imports
# -------------------------------------------------
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path
from tqdm import tqdm

# -------------------------------------------------
# 1)  File location & basic hyper-params
# -------------------------------------------------
DATA_PATH = Path("/home/apanchagatti/amogh_calibration/dataset_psi_phi.npz")

BATCH       = 64
LR          = 1e-3
EPOCHS      = 50
VAL_FRAC    = 0.10
TEST_FRAC   = 0.10
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_SEED  = 0          # reproducible splits / init

# -------------------------------------------------
# 2)  Peek at file to discover M
# -------------------------------------------------
with np.load(DATA_PATH) as npz:
    n, ch, M, _ = npz["X"].shape
    assert ch == 2, "Expected 2-channel (real/imag) input"
print(f"Detected dataset with N = {n} samples, array size M = {M}")

# -------------------------------------------------
# 3)  PyTorch Dataset wrapper
# -------------------------------------------------
class CovDataset(Dataset):
    def __init__(self, npz_path: Path):
        data = np.load(npz_path)
        self.X = torch.from_numpy(data["X"]).float()  # (N, 2, M, M)
        self.y = torch.from_numpy(data["y"]).float()  # (N, 2M)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

torch.manual_seed(TORCH_SEED)
full_ds = CovDataset(DATA_PATH)

N = len(full_ds)
n_val   = int(VAL_FRAC  * N)
n_test  = int(TEST_FRAC * N)
n_train = N - n_val - n_test
train_ds, val_ds, test_ds = random_split(
    full_ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(TORCH_SEED)
)

train_loader = DataLoader(train_ds, BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   BATCH, shuffle=False)
test_loader  = DataLoader(test_ds,  BATCH, shuffle=False)

# -------------------------------------------------
# 4)  Model definition
# -------------------------------------------------
class CalibNet(nn.Module):
    def __init__(self, M):
        super().__init__()
        self.M = M
        self.cnn = nn.Sequential(
            nn.Conv2d(2, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),  # (B, 64, 1, 1)
            nn.Flatten()
        )
        self.head = nn.Sequential(
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 2*M)
        )
        self.softplus = nn.Softplus(beta=1.5)   # positive gains

    def forward(self, x):
        z = self.cnn(x)                       # (B, 64)
        out = self.head(z)                    # (B, 2M)
        psi_raw, phi_raw = out[:, :self.M], out[:, self.M:]
        psi_hat  = self.softplus(psi_raw)     # Ψ ≥ 0
        phi_hat  = torch.tanh(phi_raw) * np.pi  # Φ ∈ (−π, π)
        return torch.cat([psi_hat, phi_hat], dim=1)

model = CalibNet(M).to(DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=LR)
crit  = nn.MSELoss()

# -------------------------------------------------
# 5)  Helper to run one epoch
# -------------------------------------------------
def run_epoch(loader, training=False):
    model.train(training)
    total_mse = 0.0
    with torch.set_grad_enabled(training):
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            if training:
                opt.zero_grad()
            y_hat = model(X)
            loss  = crit(y_hat, y)
            if training:
                loss.backward()
                opt.step()
            total_mse += loss.item() * X.size(0)
    rmse = np.sqrt(total_mse / len(loader.dataset))
    return rmse

# -------------------------------------------------
# 6)  Training loop
# -------------------------------------------------
best_val = float("inf")
for epoch in range(1, EPOCHS+1):
    train_rmse = run_epoch(train_loader, training=True)
    val_rmse   = run_epoch(val_loader,   training=False)

    if val_rmse < best_val:
        best_val = val_rmse
        torch.save(model.state_dict(), f"calibnet_M{M}.pt")

    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"train RMSE {train_rmse:.4f}  val RMSE {val_rmse:.4f}")

# -------------------------------------------------
# 7)  Final test evaluation
# -------------------------------------------------
model.load_state_dict(torch.load(f"calibnet_M{M}.pt"))
test_rmse = run_epoch(test_loader, training=False)
print(f"\nBest model test RMSE = {test_rmse:.4f}")
print(f"Model weights saved to calibnet_M{M}.pt")


Detected dataset with N = 10000 samples, array size M = 6
Epoch 01/50  train RMSE 0.3215  val RMSE 0.2613
Epoch 02/50  train RMSE 0.2346  val RMSE 0.1941
Epoch 03/50  train RMSE 0.1578  val RMSE 0.1040
Epoch 04/50  train RMSE 0.1013  val RMSE 0.0938
Epoch 05/50  train RMSE 0.0946  val RMSE 0.0946
Epoch 06/50  train RMSE 0.0899  val RMSE 0.0847
Epoch 07/50  train RMSE 0.0853  val RMSE 0.0843
Epoch 08/50  train RMSE 0.0809  val RMSE 0.0777
Epoch 09/50  train RMSE 0.0792  val RMSE 0.0770
Epoch 10/50  train RMSE 0.0760  val RMSE 0.0741
Epoch 11/50  train RMSE 0.0717  val RMSE 0.0718
Epoch 12/50  train RMSE 0.0700  val RMSE 0.0673
Epoch 13/50  train RMSE 0.0671  val RMSE 0.0689
Epoch 14/50  train RMSE 0.0663  val RMSE 0.0633
Epoch 15/50  train RMSE 0.0636  val RMSE 0.0630
Epoch 16/50  train RMSE 0.0630  val RMSE 0.0601
Epoch 17/50  train RMSE 0.0623  val RMSE 0.0583
Epoch 18/50  train RMSE 0.0604  val RMSE 0.0603
Epoch 19/50  train RMSE 0.0610  val RMSE 0.0576
Epoch 20/50  train RMSE 0.0594

/tmp/ipykernel_2498597/959260166.py:145: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"calibnet_M{M}.pt"))
